# DADS5001 Mini Project
## Government Procurement: Data Understanding and Preparation

Notebook นี้ใช้สำรวจภาพรวมข้อมูลการจัดซื้อจัดจ้างภาครัฐ
ปีงบประมาณ 2569 และเตรียมชุดข้อมูลสำหรับการวิเคราะห์เชิงสำรวจ
ในขั้นต่อไป

เนื่องจากข้อมูลต้นทางมีขนาดรวมประมาณ 4 GB จึงอ่านข้อมูลแบบ
แบ่งส่วน (chunk processing) พร้อมสรุปจำนวนรายการตามประเภทโครงการ
และสกัดเฉพาะข้อมูลงาน `จ้างก่อสร้าง` ออกมาเป็นไฟล์ใหม่

> การวิเคราะห์นี้มุ่งค้นหารูปแบบหรือสัญญาณที่ควรตรวจสอบเพิ่มเติม
> ไม่ใช่การยืนยันว่ามีการทุจริต

## วิธีรันเวอร์ชัน GitHub

เปิด Notebook นี้ใน Google Colab แล้วเลือก **Runtime → Run all** ข้อมูลจะอ่านจาก [data_for_github](https://drive.google.com/drive/folders/1ssVrUcY4TiYee9T2B0pwgr5SwvPp_lAq) โดยดาวน์โหลดเฉพาะไฟล์ที่ใช้ลงพื้นที่ชั่วคราวของ Runtime ไม่ต้องตั้งโฟลเดอร์ผลลัพธ์หรือเชื่อม Drive ส่วนตัว

- กราฟและตารางแสดงใน Notebook ไม่มีการส่งออก CSV/PNG/SVG ไปยัง Drive หรือเครื่องของผู้อ่าน
- ไฟล์ข้อมูล ฟอนต์ แผนที่ และ CSV พักที่จำเป็นอยู่ใน Runtime เท่านั้น เซลล์สุดท้ายลบไฟล์พักและเก็บ DataFrame ไว้ให้วิเคราะห์ต่อ หากหยุดกลางทางให้รัน `cleanup_runtime()` เมื่อไม่ต้องใช้ไฟล์พักแล้ว
- รันแต่ละ Notebook ใน Runtime แยกกันเพื่อลด RAM; หาก RAM ใช้ถึง 80% โค้ดจะหยุดพร้อมคำแนะนำ แทนการฝืนประมวลผล
- กรณี Drive จำกัดการดาวน์โหลด ให้ดาวน์โหลดข้อมูลเองและระบุ `LOCAL_DATA_DIR` ในเซลล์ตั้งค่า
- สูตร เกณฑ์ ตัวกรอง และลำดับข้อมูลคงตามไฟล์ต้นฉบับ อ่านข้อแตกต่างระหว่าง Notebook และรายการไฟล์ใน [DATA_FILES.md](../DATA_FILES.md)

ผลลัพธ์เดิมที่ฝังในไฟล์ถูกล้างเพื่อให้ผู้อ่านเห็นผลจากการรันครั้งใหม่


In [ ]:
# GitHub / Colab: ดาวน์โหลดข้อมูลเฉพาะที่ใช้ลงพื้นที่ชั่วคราว ไม่ต้อง Mount Drive
# เปลี่ยนเป็น Path โฟลเดอร์ข้อมูลที่มีอยู่แล้วได้ เพื่อไม่ต้องดาวน์โหลดซ้ำ
LOCAL_DATA_DIR = None
import atexit
import gc
import hashlib
import importlib.util
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

# ติดตั้งเฉพาะแพ็กเกจที่ยังไม่มี; subprocess นี้รอจนจบ ไม่มี worker ค้าง
_packages = {"pandas": "pandas>=2.2,<3", "numpy": "numpy", "matplotlib": "matplotlib", "seaborn": "seaborn", "geopandas": "geopandas", "psutil": "psutil", "gdown": "gdown", "IPython": "ipython"}
_missing_packages = [requirement for module, requirement in _packages.items() if importlib.util.find_spec(module) is None]
if _missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing_packages], check=True)
import psutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gdown
from IPython.display import display

def check_memory(stage="ประมวลผล"):
    # หยุดอย่างชัดเจนเมื่อ RAM สูง แทนการฝืนจน Runtime หลุด
    if psutil.virtual_memory().percent >= 80:
        gc.collect()
        if psutil.virtual_memory().percent >= 80:
            raise MemoryError(f"RAM ใช้เกิน 80% ก่อน {stage}; กรุณา Restart runtime หรือใช้ High-RAM แล้ว Run all ใหม่")

# รัน setup ซ้ำจะปิดพื้นที่ชั่วคราวของ Notebook รอบก่อน
if "_github_runtime" in globals():
    _github_runtime.cleanup()
_github_runtime = tempfile.TemporaryDirectory(prefix="procurement_github_")
atexit.register(_github_runtime.cleanup)
RUNTIME_DIR = Path(_github_runtime.name)
DATA_DIR = RUNTIME_DIR / "inputs"
DATA_DIR.mkdir()
_initial_figure_numbers = set(plt.get_fignums())

DATA_FILES = {
    "07_lower10_project_budget_2567.csv": {
        "id": "1YQpAhUcghJ9gM6eIC6qBZGe6WcVuKyWR",
        "size": 4169
    },
    "07_lower10_project_budget_2568.csv": {
        "id": "1aNYIMOH7Da8VvVhXI1XQigBYRNWk9IHj",
        "size": 5098
    },
    "07_lower10_project_budget_2569.csv": {
        "id": "1tU1V9n2ysJmXxvmnm_SPz7n7Bhbz2B5_",
        "size": 4371
    },
    "07_top10_project_budget_2567.csv": {
        "id": "1bkMYSpLM4UKivKCHe7S8Dsho0BaLgCdJ",
        "size": 5619
    },
    "07_top10_project_budget_2568.csv": {
        "id": "1xsItnomfe9q4A9NYJ5U_lR0764mZMEE4",
        "size": 5682
    },
    "07_top10_project_budget_2569.csv": {
        "id": "1ZwOBFVQkTRAnazjoeXzr3jkpSGFl2bkI",
        "size": 6385
    },
    "2569-egp-contract-1.csv": {
        "id": "1ygmuKsvVjNpjfg0k94Olcuox8efDP3cY",
        "size": 615601896
    },
    "2569-egp-contract-2.csv": {
        "id": "1DUAh8zAtUUV4Hm2yRplCryUOQDFb3Ad8",
        "size": 542097077
    },
    "2569-egp-contract-3.csv": {
        "id": "1fD_6GqtIY7LUempYqId0qIEldcJd2moM",
        "size": 529193616
    },
    "2569-egp-contract-4.csv": {
        "id": "19C4Ak8XNMvm2IBsr7TVpfexZE_1WBke5",
        "size": 526729581
    },
    "2569-egp-contract-5.csv": {
        "id": "1nsV7vqF8Um-QWKX_Q-kEQq6Z1BFDbK_y",
        "size": 521850128
    },
    "2569-egp-contract-6.csv": {
        "id": "1-0zhDZjF2chjeXuPoSVGQWNYtp8fVIzt",
        "size": 522697146
    },
    "2569-egp-contract-7.csv": {
        "id": "1gl9sx_1ve-n7e9M_mQaj0A7mMnk2P0yb",
        "size": 517466336
    },
    "2569-egp-contract-8.csv": {
        "id": "1z0icf65DY2e7WPmc_qV5PPLdUVyT1wdR",
        "size": 455588827
    },
    "construction_contract_review_indicators_2569.csv": {
        "id": "1nYgjub4kfLTyX_9gnz3dM2GZWUM67wRJ",
        "size": 247613515
    },
    "construction_contract_supplier_study_scope_2569.csv": {
        "id": "1dXH9N4ukZAabRQwl9nkQ-KidB1e80ZAh",
        "size": 243311179
    },
    "priority_review_contracts_2569.csv": {
        "id": "1prDQwSgkNCFqarTx0rkFuQK12Q5J2MT_",
        "size": 108598
    },
    "project_overview_2567.csv": {
        "id": "1hEr6tXWRBgEM-uK0dOEKvzIhBdKzHtnx",
        "size": 1685455822
    },
    "project_overview_2568.csv": {
        "id": "1WVh1etQtH45AemJE4J7mBCDOOT9QmQff",
        "size": 1448461110
    },
    "project_overview_2569.csv": {
        "id": "1h3Oah1ARYenycAJnLa1DTefzjMryQuJ5",
        "size": 1227718817
    },
    "repeated_near_500k_agency_supplier_2569.csv": {
        "id": "10fT6PqZbBa5mS4GEHHEOf8_qIeASToQk",
        "size": 364037
    }
}

def data_file(name):
    # ใช้ไฟล์ในเครื่องก่อนถ้าผู้ใช้ระบุโฟลเดอร์ โดยไม่แก้ไขไฟล์ต้นฉบับ
    if LOCAL_DATA_DIR is not None:
        candidate = Path(LOCAL_DATA_DIR) / name
        if candidate.is_file():
            return candidate
    if name not in DATA_FILES:
        raise FileNotFoundError(f"ยังไม่มีข้อมูล {name} ในชุดที่แชร์ กรุณาตรวจ DATA_FILES หรือ LOCAL_DATA_DIR")
    item = DATA_FILES[name]
    target = DATA_DIR / name
    if target.is_file():
        return target
    required_bytes = int(item["size"])
    if shutil.disk_usage(DATA_DIR).free < required_bytes + 512 * 1024**2:
        raise OSError(f"พื้นที่ชั่วคราวไม่พอสำหรับ {name}")
    partial = target.with_suffix(target.suffix + ".part")
    print(f"อ่านข้อมูล {name} ({required_bytes / 1024**2:,.1f} MB)")
    try:
        result = gdown.download(id=item["id"], output=str(partial), quiet=False)
        if result is None or not partial.is_file() or partial.stat().st_size != required_bytes:
            raise IOError(f"ดาวน์โหลด {name} ไม่ครบ หรือไฟล์ใน Drive เปลี่ยนรุ่น")
        if item.get("md5"):
            digest = hashlib.md5()
            with partial.open("rb") as stream:
                for block in iter(lambda: stream.read(4 * 1024**2), b""):
                    digest.update(block)
            if digest.hexdigest() != item["md5"]:
                raise IOError(f"Checksum ไม่ตรงสำหรับ {name}")
        partial.replace(target)
    except Exception as exc:
        partial.unlink(missing_ok=True)
        raise RuntimeError(f"อ่าน {name} ไม่สำเร็จ: ตรวจสิทธิ์ Anyone with the link ของโฟลเดอร์ข้อมูล หรือดาวน์โหลดเองแล้วกำหนด LOCAL_DATA_DIR") from exc
    return target

def release_download(path):
    # ลบเฉพาะสำเนาที่ Notebook นี้ดาวน์โหลด ไม่ลบไฟล์ใน LOCAL_DATA_DIR
    path = Path(path)
    if path.parent == DATA_DIR:
        path.unlink(missing_ok=True)

def cleanup_runtime():
    # เก็บ DataFrame ผลวิเคราะห์ไว้ให้ย้อนกลับมาดูต่อได้
    for number in set(plt.get_fignums()) - _initial_figure_numbers:
        plt.close(number)
    _github_runtime.cleanup()
    gc.collect()
    print("ปิดรูปและลบไฟล์พักของ Notebook แล้ว; DataFrame ผลวิเคราะห์ยังอยู่ใน RAM")

check_memory("เริ่ม Notebook")



In [ ]:
# GitHub: ใช้ข้อมูลจากลิงก์ที่แชร์ ไม่ต้อง Mount Drive


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

sns.set_theme(style='whitegrid')

# รายชื่อไฟล์จริงจาก snapshot ที่แชร์; ดาวน์โหลดทีละไฟล์ในเซลล์ประมวลผล
base_dir = DATA_DIR
processed_dir = RUNTIME_DIR
raw_names = sorted(n for n in DATA_FILES if n.startswith("2569-egp-contract-") and n.endswith(".csv"))
csv_files = [DATA_DIR / name for name in raw_names]
construction_path = RUNTIME_DIR / 'construction_contracts_2569.csv'
if not csv_files:
    raise FileNotFoundError("ไม่พบไฟล์ต้นทางปี 2569 ใน manifest")
print(f'CSV files found: {len(csv_files)}')


## 1. Data Preview

เริ่มจากอ่านข้อมูลตัวอย่างจากไฟล์แรก เพื่อทำความเข้าใจโครงสร้าง
ชื่อคอลัมน์ และลักษณะข้อมูล ก่อนประมวลผลไฟล์ทั้งหมด

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
csv_files[0] = data_file(raw_names[0])
sample_data = pd.read_csv(
    csv_files[0],
    nrows=5
)

print(f'Sample file: {csv_files[0].name}')
print(f'Number of columns: {sample_data.shape[1]}')

display(sample_data)

In [ ]:
for position, column in enumerate(
    sample_data.columns,
    start=1
):
    print(f'{position:02d}. {column}')

## 2. Project Type Overview and Data Extraction

ข้อมูลทั้งหมดจะถูกอ่านแบบแบ่งส่วน เพื่อทำงานสองอย่างในรอบเดียว:

1. นับจำนวนรายการตามประเภทโครงการ เพื่อดูภาพรวมของข้อมูล
2. สกัดรายการงาน `จ้างก่อสร้าง` ทุกคอลัมน์ออกมาเป็นไฟล์ใหม่

ในขั้นนี้ใช้คำว่า “จำนวนรายการ” เนื่องจากหนึ่งโครงการอาจมี
หลายสัญญาและปรากฏได้มากกว่าหนึ่งแถว

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
project_type_column = 'ชื่อประเภทโครงการ'
chunk_size = 100_000

type_counts_list = []
processing_results = []

first_write = True

for file_number, file_path in enumerate(
    csv_files,
    start=1
):
    print(
        f'Processing {file_number}/{len(csv_files)}: '
        f'{file_path.name}'
    )

    file_path = data_file(file_path.name)
    check_memory("อ่านไฟล์ต้นทาง")
    total_rows = 0
    construction_rows = 0

    reader = pd.read_csv(
        file_path,
        chunksize=chunk_size,
        low_memory=False
    )

    try:
        for chunk in reader:
            check_memory("คัดงานก่อสร้าง")
            chunk.columns = chunk.columns.str.strip()
            total_rows += len(chunk)

            project_type = (
                chunk[project_type_column]
                .astype('string')
                .str.strip()
                .fillna('ไม่ระบุ')
            )

            type_counts_list.append(
                project_type.value_counts()
            )

            construction_chunk = chunk.loc[
                project_type.eq('จ้างก่อสร้าง')
            ].copy()

            construction_rows += len(
                construction_chunk
            )

            if not construction_chunk.empty:
                construction_chunk['source_file'] = (
                    file_path.name
                )

                construction_chunk.to_csv(
                    construction_path,
                    mode='w' if first_write else 'a',
                    header=first_write,
                    index=False,
                    encoding='utf-8-sig'
                )

                first_write = False

    finally:
        reader.close()
    release_download(file_path)
    gc.collect()

    processing_results.append({
        'file_name': file_path.name,
        'total_rows': total_rows,
        'construction_rows': construction_rows
    })

    print(
        f'  Total rows: {total_rows:,} | '
        f'Construction rows: {construction_rows:,}'
    )

processing_summary = pd.DataFrame(
    processing_results
)

project_type_counts = (
    pd.concat(
        type_counts_list,
        axis=1
    )
    .fillna(0)
    .sum(axis=1)
    .astype('int64')
    .sort_values(ascending=False)
    .rename('record_count')
    .reset_index()
    .rename(
        columns={
            'index': project_type_column
        }
    )
)

total_records = (
    processing_summary['total_rows'].sum()
)

total_construction_records = (
    processing_summary[
        'construction_rows'
    ].sum()
)

construction_pct = (
    total_construction_records
    / total_records
    * 100
)

display(processing_summary)

print(f'Total records: {total_records:,}')
print(
    f'Construction records: '
    f'{total_construction_records:,}'
)
print(
    f'Construction share: '
    f'{construction_pct:.2f}%'
)
print("สร้างข้อมูลก่อสร้างในพื้นที่ชั่วคราวแล้ว")

### Processing Result

ข้อมูลต้นทางปีงบประมาณ 2569 ประกอบด้วย 8 ไฟล์ รวมทั้งหมด
3,964,924 รายการ

เมื่อกรองด้วย `ชื่อประเภทโครงการ = 'จ้างก่อสร้าง'`
พบข้อมูลจำนวน 180,079 รายการ คิดเป็น 4.54% ของข้อมูลทั้งหมด
และได้บันทึกข้อมูลกลุ่มดังกล่าวเป็นไฟล์
`construction_contracts_2569.csv` สำหรับใช้วิเคราะห์ต่อ

จำนวนดังกล่าวเป็นจำนวนแถวหรือรายการในข้อมูล ไม่ใช่จำนวนโครงการ
ที่ไม่ซ้ำ เนื่องจากหนึ่งโครงการอาจมีมากกว่าหนึ่งสัญญา
หรือมากกว่าหนึ่งผู้ชนะการเสนอราคา

In [ ]:
project_type_counts['record_pct'] = (
    project_type_counts['record_count']
    .div(
        project_type_counts[
            'record_count'
        ].sum()
    )
    .mul(100)
)

display(project_type_counts)

In [ ]:
major_types = (
    project_type_counts
    .head(4)
    .sort_values(
        'record_pct',
        ascending=True
    )
    .copy()
)

label_map = {
    'ซื้อ': 'Purchase',
    'จ้างทำของ/จ้างเหมาบริการ': 'Service',
    'จ้างก่อสร้าง': 'Construction',
    'เช่า': 'Rental'
}

major_types['project_type_en'] = (
    major_types['ชื่อประเภทโครงการ']
    .map(label_map)
)

colors = [
    '#E67E22'
    if project_type == 'จ้างก่อสร้าง'
    else '#8FA3B8'
    for project_type
    in major_types['ชื่อประเภทโครงการ']
]

fig, ax = plt.subplots(
    figsize=(10, 5)
)

bars = ax.barh(
    major_types['project_type_en'],
    major_types['record_pct'],
    color=colors
)

ax.bar_label(
    bars,
    labels=[
        f'{value:.2f}%'
        for value
        in major_types['record_pct']
    ],
    padding=4
)

ax.set_title(
    'Share of Procurement Records by Major Project Type'
)
ax.set_xlabel('Share of records (%)')
ax.set_ylabel('Project type')
ax.set_xlim(
    0,
    major_types['record_pct'].max() * 1.12
)

ax.spines[
    ['top', 'right', 'left']
].set_visible(False)

plt.tight_layout()
plt.show()

### Project Type Overview

รายการจัดซื้อจัดจ้างส่วนใหญ่เป็นประเภทงาน `ซื้อ` คิดเป็น 62.90%
รองลงมาคือ `จ้างทำของ/จ้างเหมาบริการ` คิดเป็น 31.34%

`จ้างก่อสร้าง` มีจำนวน 180,079 รายการ หรือ 4.54% ของข้อมูลทั้งหมด
เป็นประเภทโครงการที่มีจำนวนรายการมากเป็นอันดับ 3 แม้มีสัดส่วน
น้อยกว่าสองประเภทหลักอย่างชัดเจน แต่ยังมีจำนวนข้อมูลมากเพียงพอ
สำหรับการวิเคราะห์เชิงสำรวจในหลายมิติ

โครงการจ้างก่อสร้างยังมีข้อมูลเกี่ยวกับวงเงินงบประมาณ ราคากลาง
ราคาที่ตกลง วิธีจัดซื้อ หน่วยงาน ผู้ชนะ และรายละเอียดสัญญา
จึงเหมาะสำหรับศึกษาการกระจายของมูลค่า ความแตกต่างระหว่างราคา
การกระจุกตัวของผู้รับจ้าง และรายการที่มีลักษณะแตกต่างจากกลุ่ม

### Decision

การวิเคราะห์ขั้นต่อไปจะจำกัดขอบเขตเฉพาะงาน `จ้างก่อสร้าง`
และใช้ไฟล์ `construction_contracts_2569.csv` ที่สกัดไว้แล้ว
โดยไม่ต้องอ่านไฟล์ต้นทางขนาดประมาณ 4 GB ซ้ำ

## 3. Construction Dataset Validation

ก่อนจบขั้นตอนเตรียมข้อมูล จะโหลดไฟล์จ้างก่อสร้างที่สร้างขึ้น
เพื่อตรวจสอบจำนวนแถว จำนวนคอลัมน์ และตัวอย่างข้อมูล
ให้แน่ใจว่าไฟล์พร้อมสำหรับใช้ใน Notebook EDA

In [ ]:
# ตรวจ RAM ก่อนอ่านหรือรวมข้อมูลขนาดใหญ่
check_memory()
construction_data = pd.read_csv(
    construction_path,
    low_memory=False
)

print(
    f'Shape: '
    f'{construction_data.shape}'
)

display(
    construction_data.head()
)
# โหลดเข้า RAM แล้ว ลบเฉพาะ CSV พักที่ Notebook สร้าง
construction_path.unlink(missing_ok=True)


In [ ]:
validation_summary = pd.Series({
    'Rows': len(construction_data),
    'Columns': construction_data.shape[1],
    'Unique project IDs': (
        construction_data[
            'รหัสโครงการ'
        ].nunique()
    ),
    'Unique contract numbers': (
        construction_data[
            'เลขที่สัญญา'
        ].nunique()
    ),
    'Missing project IDs': (
        construction_data[
            'รหัสโครงการ'
        ].isna().sum()
    ),
    'Missing contract numbers': (
        construction_data[
            'เลขที่สัญญา'
        ].isna().sum()
    ),
    'Exact duplicate rows': (
        construction_data
        .drop(columns='source_file')
        .duplicated()
        .sum()
    )
})

display(
    validation_summary.to_frame(
        name='value'
    )
)

display(
    construction_data[
        'ชื่อประเภทโครงการ'
    ].value_counts(
        dropna=False
    )
)

In [ ]:
project_row_counts = (
    construction_data[
        'รหัสโครงการ'
    ]
    .value_counts()
)

print(
    'Projects with more than one row:',
    (project_row_counts > 1).sum()
)

print(
    'Maximum rows per project:',
    project_row_counts.max()
)

print(
    '\nMost frequent contract numbers:'
)

display(
    construction_data[
        'เลขที่สัญญา'
    ]
    .value_counts(
        dropna=False
    )
    .head(10)
)

repeated_project_ids = (
    project_row_counts.loc[
        project_row_counts > 1
    ]
    .head(5)
    .index
)

display(
    construction_data.loc[
        construction_data[
            'รหัสโครงการ'
        ].isin(
            repeated_project_ids
        ),
        [
            'รหัสโครงการ',
            'ชื่อโครงการจัดซื้อจัดจ้าง',
            'ชื่อผู้ชนะการเสนอราคา',
            'เลขที่สัญญา',
            'วงเงินงบประมาณในสัญญา (บาท)'
        ]
    ]
    .sort_values(
        [
            'รหัสโครงการ',
            'เลขที่สัญญา'
        ]
    )
)

## 4. Data Preparation Summary

ข้อมูลจัดซื้อจัดจ้างภาครัฐปีงบประมาณ 2569 มีทั้งหมด
3,964,924 รายการ โดยงาน `ซื้อ` เป็นประเภทที่พบมากที่สุด
คิดเป็น 62.90% รองลงมาคืองาน `จ้างทำของ/จ้างเหมาบริการ`
คิดเป็น 31.34%

งาน `จ้างก่อสร้าง` มีจำนวน 180,079 รายการ คิดเป็น 4.54%
ของข้อมูลทั้งหมด และเป็นประเภทโครงการที่มีจำนวนรายการ
มากเป็นอันดับ 3 ข้อมูลกลุ่มนี้จึงมีขนาดเพียงพอสำหรับ
การวิเคราะห์เชิงสำรวจในหลายมิติ

หลังจากสกัดข้อมูลงานจ้างก่อสร้าง พบว่า:

- มีข้อมูลทั้งหมด 180,079 แถว และ 29 คอลัมน์
- มี `รหัสโครงการ` ที่ไม่ซ้ำจำนวน 178,978 โครงการ
- ไม่พบ `รหัสโครงการ` ที่เป็นค่าว่าง
- ไม่พบข้อมูลที่ซ้ำกันทั้งแถว
- มี 524 โครงการที่ปรากฏมากกว่าหนึ่งแถว
- หนึ่งโครงการปรากฏได้สูงสุด 15 แถว

การที่หนึ่งโครงการปรากฏหลายแถวอาจเกิดจากโครงการหนึ่งมี
หลายสัญญา หลายผู้ชนะการเสนอราคา หรือมีผู้รับจ้างในลักษณะ
กิจการร่วมค้า จึงไม่ควรลบแถวเหล่านี้ในขั้นเตรียมข้อมูล

นอกจากนี้ `เลขที่สัญญา` ไม่ได้เป็นรหัสที่ไม่ซ้ำสำหรับข้อมูลทั้งหมด
ตัวอย่างเช่น `1/2569` ถูกใช้ในหลายหน่วยงาน ดังนั้นจึงไม่สามารถ
ใช้จำนวนค่าที่ไม่ซ้ำของ `เลขที่สัญญา` เป็นจำนวนสัญญาทั้งหมดได้โดยตรง

### Analysis Decision

การวิเคราะห์ในขั้นต่อไปจะแยกออกเป็นสองระดับ:

1. **Project-level analysis**  
   ใช้ `รหัสโครงการ` เป็นหน่วยวิเคราะห์ สำหรับศึกษาจำนวนโครงการ
   งบประมาณ ราคากลาง ราคาที่ตกลง วิธีจัดซื้อ หน่วยงาน และพื้นที่

2. **Contract and supplier-level analysis**  
   ใช้ข้อมูลทุกแถวสำหรับศึกษาสัญญา ผู้ชนะการเสนอราคา
   และโครงการที่มีหลายรายการ โดยจะไม่ถือว่า `เลขที่สัญญา`
   เป็นรหัสที่ไม่ซ้ำในระดับประเทศ

ข้อมูลที่เตรียมแล้วถูกบันทึกเป็น
`construction_contracts_2569.csv` และจะใช้เป็นแหล่งข้อมูลหลัก
ของ Notebook การวิเคราะห์เชิงสำรวจ โดยไม่ต้องอ่านไฟล์ต้นทาง
ขนาดประมาณ 4 GB ซ้ำ

In [ ]:
# คืนพื้นที่ชั่วคราวและปิดรูป โดยเก็บ DataFrame ผลวิเคราะห์ไว้
cleanup_runtime()
